In [1]:
import preprocess
import run_cicero
import annotate_peaks

In [2]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread

In [3]:
counts = mmread("../make_cicero_cds_R_Objs/full_matrix.mtx").T.tocsr()
obs_names = pd.read_csv("../make_cicero_cds_R_Objs/full_meta.csv", header = 0, index_col = 0)
var_names = pd.read_csv("../make_cicero_cds_R_Objs/full_matrix_rownames.csv", header = 0, index_col = 1)
var_names.drop("Unnamed: 0", axis = 1, inplace = True)
# obs_names.drop("Unnamed: 0", axis = 1, inplace = True)

In [4]:
adata = sc.AnnData(counts, obs = obs_names, var = var_names)

In [5]:
adata

AnnData object with n_obs × n_vars = 37584 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group'

In [6]:
adata = preprocess.preprocess_cicero(adata)

preprocess.py: 2025-04-16 05:41:29 WARNING  Counts layers not found. Coppying X to counts
preprocess.py: 2025-04-16 05:41:29 INFO     Estimaing Size Factors and Normalizing Data
preprocess.py: 2025-04-16 05:41:36 INFO     Finished Size Factors and Normallization storing normalized sparse matrix into 'data'
preprocess.py: 2025-04-16 05:41:36 INFO     Running TF-IDF
preprocess.py: 2025-04-16 05:41:40 INFO     Finsihed TF-IDF
preprocess.py: 2025-04-16 05:41:40 INFO     Running TruncatedSVD
preprocess.py: 2025-04-16 05:44:09 INFO     Finsihed TruncatedSVD
preprocess.py: 2025-04-16 05:44:09 INFO     Running UMAP
preprocess.py: 2025-04-16 05:44:09 INFO     Finsihed UMAP


In [7]:
cicero_adata = preprocess.make_cicero_adata(adata, k = 50)

preprocess.py: 2025-04-16 05:44:09 INFO     Using adata OBSM X_umap to agregate cells
preprocess.py: 2025-04-16 05:44:09 INFO     Calculating overlap
preprocess.py: 2025-04-16 05:44:11 INFO     Generating Pseudobulk cicero observations with seed: 0 and k: 50
preprocess.py: 2025-04-16 05:44:27 INFO     Reached Maximum itterations in pseudobulk observation generation. Consider increasing 'max_itterations'
preprocess.py: 2025-04-16 05:44:27 INFO     Found 4502 good choices
preprocess.py: 2025-04-16 05:44:27 INFO     Finished calculating overlap
preprocess.py: 2025-04-16 05:44:27 INFO     Aggregating Cells
preprocess.py: 2025-04-16 05:44:43 INFO     Finished Aggregating Cells


In [8]:
# Preprocess the index strings, assuming format "chr-start-end" (e.g., "chr-0-1923") 
# Also sorted from lower to higher
temp_df = pd.DataFrame([x.split("-") for x in cicero_adata.var.index])
cicero_adata.var["Chromosome"] = temp_df.iloc[:, 0].values
cicero_adata.var["Start"] = temp_df.iloc[:, 1].astype(int).values
cicero_adata.var["End"] = temp_df.iloc[:, 2].astype(int).values
cicero_adata.var["Mean"] = (cicero_adata.var["Start"] + cicero_adata.var["End"])/2

In [9]:
#Usually sorted, if not sort
chromosomes_ordered = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8',
       'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
       'chr16', 'chr17', 'chr18', 'chr19', 'chrX', 'chrY', 'GL456211.1',
       'GL456216.1', 'GL456233.1', 'JH584295.1', 'JH584304.1'] #doesn't actually matter
temp_vars = []
for chromosome in chromosomes_ordered:
    chromosome_var = cicero_adata.var[cicero_adata.var["Chromosome"] == chromosome]
    chromosome_var.sort_values("Start", ascending = False) #lower to higher
    temp_vars.append(chromosome_var)
cicero_adata.var = pd.concat(temp_vars, axis = 0)

In [10]:
cicero_adata.var

,Chromosome,Start,End,Mean
x,,,,
chr1-3119740-3120239,chr1,3119740,3120239,3119989.5
chr1-3121226-3121725,chr1,3121226,3121725,3121475.5
chr1-3155081-3155580,chr1,3155081,3155580,3155330.5
chr1-3203852-3204351,chr1,3203852,3204351,3204101.5
chr1-3210121-3210620,chr1,3210121,3210620,3210370.5
...,...,...,...,...
JH584304.1-67266-67765,JH584304.1,67266,67765,67515.5
JH584304.1-67886-68385,JH584304.1,67886,68385,68135.5
JH584304.1-68747-69246,JH584304.1,68747,69246,68996.5


In [11]:
cicero_adata

AnnData object with n_obs × n_vars = 4502 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group', 'aggregate_obs_names'
    var: 'Chromosome', 'Start', 'End', 'Mean'

In [12]:
"""
probably from inverse_covriance (skggm)
from: Starting distance_parameter_estimation
/home/twoo/miniconda3/envs/rapids_SC/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
"""
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

In [13]:
cons = run_cicero.run_cicero(cicero_adata)

run_cicero.py: 2025-04-16 05:44:45 INFO     Generating Windows


run_cicero.py: 2025-04-16 05:44:45 INFO     Starting distance_parameter_estimation
run_cicero.py: 2025-04-16 05:52:23 INFO     Starting multiprocessing pool for estimate_distance_parameter_parallel with 96 processes
run_cicero.py: 2025-04-16 06:02:10 INFO     Finished distance_parameter_estimation
run_cicero.py: 2025-04-16 06:02:10 INFO     Starting generate_cicero_models
run_cicero.py: 2025-04-16 06:03:03 INFO     Starting Cicero with 96 processes
run_cicero.py: 2025-04-16 06:03:16 WARNING  diag(V) had non-positive or NA entries; the non-finite result may be dubious
run_cicero.py: 2025-04-16 06:04:22 WARNING  diag(V) had non-positive or NA entries; the non-finite result may be dubious
run_cicero.py: 2025-04-16 06:04:29 INFO     Finished generate_cicero_models
run_cicero.py: 2025-04-16 06:04:29 INFO     Starting assemble_connections
run_cicero.py: 2025-04-16 06:04:53 INFO     Finished assemble_connections


In [14]:
cons = cons[cons["coaccess_score"] >= 0.2]
cons

,Peak1,Peak2,coaccess_score
1,GL456216.1-16175-16674,GL456216.1-16819-17318,0.999969
12,GL456216.1-31947-32446,GL456216.1-32548-33047,0.988766
19,GL456216.1-33752-34251,GL456216.1-34703-35202,0.986925
30,GL456233.1-109964-110463,GL456233.1-110707-111206,0.255338
92,JH584304.1-21645-22144,JH584304.1-22289-22788,0.994234
...,...,...,...
9164623,chrY-90804920-90805419,chrY-90808559-90809058,0.745941
9164624,chrY-90804920-90805419,chrY-90810437-90810936,0.345665
9164626,chrY-90807503-90808002,chrY-90808559-90809058,0.808222
9164627,chrY-90807503-90808002,chrY-90810437-90810936,0.374526


In [21]:
tss = pd.read_csv("./gencode.vM23.annotation.gff3.gz.tss.bed", sep = "\t", header = None, names = ["Chromosome", "TSS"])
genes_df = pd.read_csv("./gencode.vM23.annotation.gff3.gz.gene.bed", sep = "\t", header = None, names = ["chr", "start", "end", "gene_name"])

cons = annotate_peaks.expand_cons(cons)
cons = annotate_peaks.annotate_distal_proximal(cons, tss)
cons = annotate_peaks.annotate_genes(cons, genes_df)

In [26]:
cons["Linkage_Type"].value_counts()/cons.shape[0] * 100

Linkage_Type
Distal-to-Distal        51.189879
Distal-to-Proximal      17.385238
Proximal-to-Distal      17.219280
Proximal-to-Proximal    14.205604
Name: count, dtype: float64

In [23]:
n_unique_genes = len(set(cons["Peak_2_Gene_Annotation"].values).union(set(cons["Peak_1_Gene_Annotation"].values))) - 2
pct_exons = (pd.notna(cons["Peak_1_Gene_Annotation"]).sum() + pd.notna(cons["Peak_2_Gene_Annotation"]).sum())/(cons.shape[0] * 2) * 100
print(f"Number of Unique Exon Genes: {n_unique_genes}")
print(f"Percent Peaks Exons: {pct_exons:.2f}%")

Number of Unique Exon Genes: 16515
Percent Peaks Exons: 32.27%
